# 📝 PREP_02a — Membuat Set Uji Independen Tambahan
Notebook ini **hanya dijalankan sekali**. Hasilnya:
- `anotasi_latih_700.xlsx` — salinan aman data latih (700 judul)
- `anotasi_uji_tambahan.xlsx` — **±330 judul uji** yang Anda labeli **tanpa saran apa pun**
- `kunci_uji_tambahan.csv` — strata & bobot sampling (**jangan dibuka** sebelum anotasi selesai)

Ada pengaman: bila file anotasi uji sudah ada, notebook berhenti dan **tidak menimpa** apa pun.

In [ ]:
# ════ SEL 1 : PATH & PENGAMAN ════
import shutil, pandas as pd, numpy as np
from pathlib import Path

ROOT = Path(r"E:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026")
SENT = ROOT / "02_preprocessing" / "sentimen_indobert" / "output_sentimen"

PATH_BERITA = ROOT / "03_dataset" / "final" / "berita_gabungan_semua_sumber_FINAL.csv"
PATH_LAMA   = SENT / "anotasi_sentimen_VALID_gabungan.csv"   # 859 judul lama (termasuk 172 judul uji Label_A)
PATH_SUMBER = SENT / "anotasi_tambahan.xlsx"                 # 700 judul latih (dibuat sebelumnya)
PATH_LATIH  = SENT / "anotasi_latih_700.xlsx"                # salinan aman data latih
PATH_UJI    = SENT / "anotasi_uji_tambahan.xlsx"             # yang Anda isi
PATH_KUNCI  = SENT / "kunci_uji_tambahan.csv"                # jangan dibuka sebelum selesai

assert not PATH_UJI.exists(), f'⛔ {PATH_UJI.name} sudah ada — notebook dihentikan agar isinya tidak tertimpa.'
for p in [PATH_BERITA, PATH_LAMA]:
    assert p.exists(), f'File tidak ditemukan: {p}'

# Salinan aman data latih
if PATH_LATIH.exists():
    print('ℹ️ anotasi_latih_700.xlsx sudah ada — dipakai apa adanya.')
else:
    assert PATH_SUMBER.exists(), f'File data latih tidak ditemukan: {PATH_SUMBER}'
    shutil.copy2(PATH_SUMBER, PATH_LATIH)
    print('✅ Disalin:', PATH_SUMBER.name, '→', PATH_LATIH.name)

In [ ]:
# ════ SEL 2 : LABEL LEKSIKON (hanya untuk stratifikasi, tidak ikut ke file anotasi) ════
KATA_POSITIF = {
 'naik','meningkat','tumbuh','menguat','melonjak','melesat','meroket','melejit',
 'terbang','rebound','rally','bullish','surplus','profit','keuntungan','rekor',
 'tertinggi','dividen','buyback','ekspansi','akuisisi','ipo','kontrak baru','optimis','pemulihan',
 'solid','cuan','top gainers','gainers','positif','cemerlang','sukses','raih',
 'menguntungkan','ekspor naik','produksi naik','harga batu bara naik','potensi cuan',
 'laba naik','laba melonjak','laba tumbuh','laba meningkat','laba melesat','catat laba',
 'untung besar','saham diborong','banyak dibeli','laba bersih naik','laba bersih tumbuh',
 'laba bersih melonjak','pendapatan naik','pendapatan tumbuh',
}
KATA_NEGATIF = {
 'turun','menurun','melemah','anjlok','ambles','jeblok','ambruk','terjun','jatuh',
 'merosot','terpangkas','rugi','kerugian','defisit','minus','bangkrut','pailit','default',
 'gagal bayar','delisting','koreksi','tertekan','bearish','rontok','negatif','buruk','tekanan',
 'sanksi','krisis','perlambatan','kontraksi','phk','pemangkasan','profit taking','outflow',
 'net sell','jual asing','tergerus','underperform','gugatan','denda','lesu','terpuruk',
 'oversupply','permintaan turun','ekspor turun','produksi turun','harga gas mahal',
 'harga batu bara turun','larangan ekspor','resesi','khawatir',
 'laba turun','laba anjlok','laba merosot','laba tergerus','laba minus','laba menurun',
 'laba bersih turun','laba bersih anjlok','laba bersih merosot','rugi bersih',
 'pendapatan turun','pendapatan menurun','pendapatan anjlok',
 'kecelakaan','kebakaran','ledakan','mogok kerja','korupsi','skandal',
 'investor kabur','keluar bursa','penyelidikan','dugaan pelanggaran','izin dicabut',
 'downgrade','revisi turun','pangkas target harga','prospek suram','anjlok tajam',
 'turun tajam','saham nyungsep','nyungsep','tuntutan hukum','kredit macet',
}

def auto_label(teks):
    t = str(teks).lower()
    p = sum(k in t for k in KATA_POSITIF); n = sum(k in t for k in KATA_NEGATIF)
    return 'Positif' if p > n else 'Negatif' if n > p else 'Netral'

In [ ]:
# ════ SEL 3 : PILIH KANDIDAT & SAMPLING BERTINGKAT ════
berita = pd.read_csv(PATH_BERITA, encoding='utf-8-sig')
lama   = pd.read_csv(PATH_LAMA, encoding='utf-8-sig')
latih  = pd.read_excel(PATH_LATIH)

norm = lambda s: s.astype(str).str.lower().str.strip()
dipakai = set(norm(lama['Headline'])) | set(norm(latih['Headline']))
kand = berita[~norm(berita['Headline']).isin(dipakai)].copy()
kand = kand.loc[~norm(kand['Headline']).duplicated()]
kand['Strata'] = kand['Headline'].apply(auto_label)
print(f'Total berita {len(berita):,} | sudah dipakai (lama + latih) {len(dipakai):,} | kandidat uji {len(kand):,}')

SEED, N = 2026, 110
bagian = [g.sample(n=min(N, len(g)), random_state=SEED) for _, g in kand.groupby('Strata')]
uji = pd.concat(bagian).sample(frac=1, random_state=SEED).reset_index(drop=True)
uji.insert(0, 'ID', range(1, len(uji) + 1))

# Cek kebocoran
assert not norm(uji['Headline']).isin(dipakai).any(), 'Ada judul uji yang tumpang tindih dengan data lama/latih!'
print('Kebocoran ke data latih/lama: 0 ✅')

In [ ]:
# ════ SEL 4 : SIMPAN FILE ANOTASI (tanpa label apa pun) & KUNCI ════
uji['Label_Manual'] = ''
uji[['ID', 'Kode_Emiten', 'Tanggal', 'Headline', 'Label_Manual']].to_excel(PATH_UJI, index=False)

pop = kand['Strata'].value_counts()
uji['Bobot'] = uji['Strata'].map(pop / uji['Strata'].value_counts())
uji[['ID', 'Headline', 'Strata', 'Bobot']].to_csv(PATH_KUNCI, index=False, encoding='utf-8-sig')

print(f'✅ {len(uji)} judul uji → {PATH_UJI.name}')
print(f'✅ Kunci strata & bobot → {PATH_KUNCI.name} (jangan dibuka sebelum anotasi selesai)')
print('Kandidat per strata :', pop.to_dict())
print('Per emiten          :', uji['Kode_Emiten'].value_counts().to_dict())

## ✍️ Setelah notebook ini selesai
1. Buka **`anotasi_uji_tambahan.xlsx`**, isi kolom **Label_Manual** dengan `positif` / `netral` / `negatif`.
2. Label diputuskan **sendiri**, tanpa LLM, leksikon, atau model — kriteria sama dengan 172 judul uji sebelumnya.
3. Kerjakan set uji ini **sebelum** data latih `anotasi_latih_700.xlsx`.
4. Simpan (Ctrl+S) dan tutup Excel. **Jangan jalankan notebook ini lagi.**